# JAX Model Training on Cloud TPU v5e

**Interactive multi-chip parallel training on GKE Kubeflow Workspaces**

This notebook demonstrates interactive, multi-device neural network training with JAX on a Cloud TPU v5e slice (4 chips, 2x2 topology).

You can run this notebook:
1. **Directly from Desktop VS Code** using the Microsoft Jupyter extension and a remote connection token (zero file uploads needed!), or
2. **In your browser** via the Kubeflow JupyterLab UI.

### What this notebook demonstrates:
- Inspecting and discovering Cloud TPU hardware via `jax.devices()`.
- Defining a multi-layer neural network with pure JAX functions.
- Data-parallel training across all 4 TPU chips using `jax.pmap` and `jax.lax.pmean`.
- Batch sharding and training loop measuring throughput (samples/sec) across TPU cores.
- Evaluation, inference, and saving model parameters to disk.

In [ ]:
import time
import jax
import jax.numpy as jnp
import numpy as np

print(f"JAX version: {jax.__version__}")
devices = jax.devices()
print(f"Available devices ({len(devices)}): {devices}")
assert len(devices) > 0, "No accelerator devices found!"

device_kind = devices[0].device_kind
platform = devices[0].platform
print(f"Platform: {platform}")
print(f"Device kind: {device_kind}")
print(f"Local TPU chips: {jax.local_device_count()}")

## 1. Dataset Generation

To make this notebook completely self-contained and runnable without external network downloads or GCS dependencies, we generate a synthetic multi-class classification dataset with 784 features (matching 28x28 images, like MNIST / Fashion-MNIST) and 10 classes.

In [ ]:
def generate_synthetic_data(num_samples=8192, num_features=784, num_classes=10, seed=42):
    """Generates a synthetic multi-class dataset."""
    np.random.seed(seed)
    # Create class prototypes
    prototypes = np.random.randn(num_classes, num_features).astype(np.float32)
    
    # Assign random labels
    labels = np.random.randint(0, num_classes, size=(num_samples,))
    
    # Generate features with noise centered around the assigned class prototype
    noise = np.random.randn(num_samples, num_features).astype(np.float32) * 0.5
    images = prototypes[labels] + noise
    
    # One-hot encode labels
    labels_one_hot = np.eye(num_classes, dtype=np.float32)[labels]
    
    return images, labels_one_hot, labels

# Generate train and test sets
train_x, train_y_one_hot, train_y = generate_synthetic_data(num_samples=8192, seed=1)
test_x, test_y_one_hot, test_y = generate_synthetic_data(num_samples=1024, seed=2)

print(f"Train shapes: images {train_x.shape}, one-hot labels {train_y_one_hot.shape}")
print(f"Test shapes:  images {test_x.shape}, one-hot labels {test_y_one_hot.shape}")

## 2. Model Architecture & Parameter Initialization

We define a 3-layer MLP: `784 (input) -> 256 -> 128 -> 10 (classes)` with ReLU activations.
Parameters are initialized using He normal initialization.

In [ ]:
def init_network_params(layers, key):
    """Initializes weights and biases for an MLP."""
    keys = jax.random.split(key, len(layers) - 1)
    params = []
    for k, (in_dim, out_dim) in zip(keys, zip(layers[:-1], layers[1:])):
        w_key, b_key = jax.random.split(k)
        scale = jnp.sqrt(2.0 / in_dim)
        w = jax.random.normal(w_key, (in_dim, out_dim)) * scale
        b = jnp.zeros((out_dim,))
        params.append({"w": w, "b": b})
    return params

layer_sizes = [784, 256, 128, 10]
init_key = jax.random.PRNGKey(42)
params = init_network_params(layer_sizes, init_key)

for i, layer in enumerate(params):
    print(f"Layer {i+1}: weight shape {layer['w'].shape}, bias shape {layer['b'].shape}")

## 3. Forward Pass, Loss, and Accuracy Functions

The network uses ReLU non-linearities and log-softmax cross-entropy loss.

In [ ]:
def forward(params, x):
    """Forward pass through the MLP."""
    activations = x
    for layer in params[:-1]:
        outputs = jnp.dot(activations, layer["w"]) + layer["b"]
        activations = jax.nn.relu(outputs)
    final_layer = params[-1]
    logits = jnp.dot(activations, final_layer["w"]) + final_layer["b"]
    return logits

def loss_fn(params, x, y_one_hot):
    """Cross-entropy loss with log-softmax."""
    logits = forward(params, x)
    log_preds = jax.nn.log_softmax(logits)
    loss = -jnp.mean(jnp.sum(log_preds * y_one_hot, axis=-1))
    return loss

def compute_accuracy(params, x, targets):
    """Computes classification accuracy."""
    logits = forward(params, x)
    preds = jnp.argmax(logits, axis=-1)
    return jnp.mean(preds == targets)

initial_loss = loss_fn(params, train_x[:128], train_y_one_hot[:128])
initial_acc = compute_accuracy(params, train_x[:128], train_y[:128])
print(f"Initial loss: {initial_loss:.4f}, Initial accuracy: {initial_acc * 100:.2f}%")

## 4. Parallel Training Step Across All TPU Chips (`jax.pmap`)

Cloud TPU v5e pods in this configuration contain 4 independent TPU chips (`2x2` topology).

Using `jax.pmap`, we compile and execute the training step simultaneously across all 4 TPU chips.
Inside `update_step`:
- `jax.value_and_grad` computes the loss and gradients on each chip's local batch shard.
- `jax.lax.pmean(..., axis_name="devices")` averages gradients across all 4 TPU chips via the high-speed inter-chip interconnect.
- Each chip updates its local copy of parameters with SGD, keeping parameters synchronized across all chips.

In [ ]:
num_devices = jax.local_device_count()
print(f"Parallelizing training across {num_devices} local TPU devices")

# Replicate parameters across all TPU devices
replicated_params = jax.tree_util.tree_map(
    lambda x: jnp.broadcast_to(x, (num_devices,) + x.shape),
    params
)

from functools import partial

@partial(jax.pmap, axis_name="devices", in_axes=(0, 0, 0, None))
def update_step(params, x, y_one_hot, lr):
    """Parallel update step executed on each TPU device."""
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y_one_hot)
    grads = jax.lax.pmean(grads, axis_name="devices")
    loss = jax.lax.pmean(loss, axis_name="devices")
    new_params = jax.tree_util.tree_map(
        lambda p, g: p - lr * g,
        params,
        grads
    )
    return new_params, loss

print("Compiled parallel update_step with jax.pmap")

## 5. Multi-Device Training Loop

Each global batch of size 256 is split evenly into shards of 64 examples per chip (`256 / 4 = 64`).
We train for 10 epochs and monitor loss, accuracy, and training throughput (samples/sec).

In [ ]:
global_batch_size = 256
per_device_batch_size = global_batch_size // num_devices
num_epochs = 10
learning_rate = 0.05

num_train = train_x.shape[0]
num_batches = num_train // global_batch_size

print(f"Global batch size: {global_batch_size} ({per_device_batch_size} per TPU chip)")
print(f"Training for {num_epochs} epochs ({num_batches} batches/epoch)...\n")

start_time = time.time()

for epoch in range(1, num_epochs + 1):
    epoch_start = time.time()
    
    # Shuffle dataset each epoch
    perm = np.random.permutation(num_train)
    epoch_x = train_x[perm]
    epoch_y = train_y_one_hot[perm]
    
    epoch_loss = 0.0
    
    for b in range(num_batches):
        start_idx = b * global_batch_size
        end_idx = start_idx + global_batch_size
        
        # Reshape for pmap: (num_devices, per_device_batch_size, ...)
        batch_x = epoch_x[start_idx:end_idx].reshape(num_devices, per_device_batch_size, -1)
        batch_y = epoch_y[start_idx:end_idx].reshape(num_devices, per_device_batch_size, -1)
        
        replicated_params, loss_per_device = update_step(
            replicated_params, batch_x, batch_y, learning_rate
        )
        epoch_loss += float(loss_per_device[0])
    
    epoch_loss /= num_batches
    epoch_duration = time.time() - epoch_start
    samples_per_sec = num_train / epoch_duration
    
    # Evaluate using parameters from device 0
    eval_params = jax.tree_util.tree_map(lambda x: x[0], replicated_params)
    test_acc = compute_accuracy(eval_params, test_x, test_y)
    
    print(f"Epoch {epoch:2d}/{num_epochs:2d} | "
          f"Loss: {epoch_loss:.4f} | "
          f"Test Acc: {test_acc * 100:.2f}% | "
          f"Throughput: {samples_per_sec:.0f} samples/sec ({epoch_duration:.2f}s)")

total_time = time.time() - start_time
print(f"\nTraining completed in {total_time:.2f} seconds.")

## 6. Model Evaluation & Sample Predictions

Let's test inference on several test samples and compare the model's predictions with ground truth.

In [ ]:
final_params = jax.tree_util.tree_map(lambda x: x[0], replicated_params)
final_test_acc = compute_accuracy(final_params, test_x, test_y)
print(f"Final Test Accuracy: {final_test_acc * 100:.2f}%\n")

# Run inference on sample inputs
sample_inputs = test_x[:5]
sample_targets = test_y[:5]

sample_logits = forward(final_params, sample_inputs)
sample_preds = jnp.argmax(sample_logits, axis=-1)

print("Predicted vs Actual Labels:")
for i in range(5):
    match = "OK" if sample_preds[i] == sample_targets[i] else "MISMATCH"
    print(f"  Sample {i+1}: Predicted = {int(sample_preds[i])}, True = {int(sample_targets[i])} [{match}]")

## 7. Save Parameters to Disk

Save the trained model parameters to `/home/jovyan/jax_tpu_model_params.npz`.
When connected remotely from VS Code, this file is written directly onto the remote pod's persistent storage volume.

In [ ]:
flat_params = {}
for i, layer in enumerate(final_params):
    flat_params[f"w_{i}"] = np.asarray(layer["w"])
    flat_params[f"b_{i}"] = np.asarray(layer["b"])

save_path = "jax_tpu_model_params.npz"
np.savez(save_path, **flat_params)
print(f"Saved trained parameters to {save_path}")

# Verify loading from disk
loaded = np.load(save_path)
print(f"Loaded parameter keys: {list(loaded.keys())}")
print("Verification successful!")